# 🛡️ AI-Powered Cybersecurity Threat Detection System
### Network Intrusion Detection using Isolation Forest + Random Forest

---
**Dataset:** Synthetic network flows (CICIDS-2017 / UNSW-NB15 structure)  
**Models:** IsolationForest (unsupervised) + Random Forest (supervised)  
**Task:** Binary (BENIGN vs ATTACK) + Multi-class (attack type)  
**Attack types:** DoS, DDoS, PortScan, BruteForce, SQLi, XSS, BotNet, Infiltration  

---
## 📋 Table of Contents
1. Setup & Imports
2. Dataset Generation & Overview
3. Data Preprocessing & Feature Engineering
4. Model Training
5. Evaluation (Binary + Multi-class)
6. Anomaly Score Visualization
7. Threat Detection Demo
8. SOC Dashboard
9. Conclusions

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image as IPImage

from src.config import *
print('Setup complete. Attack types:', ATTACK_TYPES)

## 2. Dataset Generation & Overview

In [ ]:
from src.data_generator import generate_dataset

df = generate_dataset(n_samples=50_000,
                      save_path=os.path.join(RAW_DIR, 'network_traffic.csv'))
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print('Attack type distribution:')
print(df['attack_type'].value_counts())
print(f'\nBenign: {(df.label_binary==0).sum():,} | Attack: {(df.label_binary==1).sum():,}')

In [ ]:
from src.visualize import plot_dataset_overview
plot_dataset_overview(df)
display(IPImage(filename=os.path.join(GRAPHS_DIR, 'dataset_overview.png'), width=900))

## 3. Data Preprocessing & Feature Engineering

In [ ]:
from src.preprocessor import preprocess, ENGINEERED_FEATURES

X, y_bin, y_mul, le, scaler, df_clean = preprocess(df, fit=True)
print(f'Feature matrix shape: {X.shape}')
print(f'Features used: {ENGINEERED_FEATURES}')

## 4. Model Training

In [ ]:
from src.train import train
results = train(n_samples=50_000)
print('Training complete!')

## 5. Evaluation

In [ ]:
from src.visualize import (plot_confusion_matrix_binary, plot_confusion_matrix_multi,
                            plot_roc_curve, plot_feature_importance)
from src.models import load_binary_classifier, get_feature_importance

bin_r  = results['bin_results']
mul_r  = results['mul_results']
le_out = results['le']

plot_confusion_matrix_binary(bin_r['confusion_matrix'])
plot_confusion_matrix_multi(mul_r['confusion_matrix'], list(le_out.classes_))
plot_roc_curve(results['y_bin_test'], bin_r['y_prob'])

rf_bin = load_binary_classifier()
fi = get_feature_importance(rf_bin, ENGINEERED_FEATURES)
plot_feature_importance(fi)

print(f"Binary RF Accuracy : {bin_r['report']['accuracy']*100:.2f}%")
print(f"Binary RF AUC-ROC  : {bin_r['auc']:.4f}")
print(f"Multi-class Acc    : {mul_r['accuracy']*100:.2f}%")

In [ ]:
for fname in ['confusion_matrix_binary.png', 'roc_curve.png', 'feature_importance.png']:
    fpath = os.path.join(GRAPHS_DIR, fname)
    if os.path.exists(fpath):
        print(f'── {fname} ──')
        display(IPImage(filename=fpath, width=800))

## 6. Anomaly Score Visualization

In [ ]:
from src.models import load_isolation_forest, isolation_forest_scores
from src.visualize import plot_anomaly_scores

iso = load_isolation_forest()
scores = isolation_forest_scores(iso, results['X_test'])
plot_anomaly_scores(scores, results['y_bin_test'])
display(IPImage(filename=os.path.join(GRAPHS_DIR, 'anomaly_scores.png'), width=800))

## 7. Real-Time Threat Detection Demo

In [ ]:
from src.detector import ThreatDetector

detector = ThreatDetector()

# Test a suspicious DoS-like flow
dos_flow = {
    'flow_duration': 9500, 'fwd_packets': 520, 'bwd_packets': 3,
    'fwd_bytes': 31000, 'bwd_bytes': 400, 'flow_bytes_per_sec': 850000,
    'flow_pkts_per_sec': 55, 'fwd_iat_mean': 190, 'bwd_iat_mean': 4800,
    'pkt_len_mean': 58, 'pkt_len_std': 9, 'fin_flag_count': 0,
    'syn_flag_count': 460, 'rst_flag_count': 0, 'ack_flag_count': 1,
    'psh_flag_count': 0, 'urg_flag_count': 0, 'active_mean': 7500,
    'idle_mean': 1800, 'dst_port': 80, 'protocol': 'TCP',
    'src_ip': '10.0.0.42', 'dst_ip': '192.168.1.100'
}

result = detector.detect_flow(dos_flow)
print('DETECTION RESULT:')
for k, v in result.items():
    print(f'  {k:<20}: {v}')

In [ ]:
# Batch detection on 1000 flows
df_sample = df.sample(1000, random_state=42)
result_df = detector.detect_batch(df_sample, save_alerts=True)

print('\nTop 5 Most Dangerous Flows:')
display(detector.get_top_threats(result_df, n=5))

## 8. SOC Dashboard

In [ ]:
from src.visualize import plot_threat_timeline, plot_executive_dashboard
import json

plot_threat_timeline(df)

with open(os.path.join(REPORTS_DIR, 'training_metrics.json')) as f:
    metrics = json.load(f)

y_prob = results['bin_results']['y_prob']
sev_counts = {
    'CRITICAL': int(np.sum(y_prob > 0.95)),
    'HIGH'    : int(np.sum((y_prob > 0.80) & (y_prob <= 0.95))),
    'MEDIUM'  : int(np.sum((y_prob > 0.50) & (y_prob <= 0.80))),
    'LOW'     : int(np.sum((y_prob > 0.30) & (y_prob <= 0.50))),
    'INFO'    : int(np.sum(y_prob <= 0.30)),
}

plot_executive_dashboard(metrics, sev_counts, df)
display(IPImage(filename=os.path.join(GRAPHS_DIR, 'threat_dashboard.png'), width=1000))

## 9. Conclusions

| Model | Task | Accuracy | AUC |
|-------|------|----------|-----|
| Isolation Forest | Unsupervised anomaly | ~75-80% | — |
| Random Forest | Binary (BENIGN/ATTACK) | ~97%+ | ~0.99 |
| Random Forest | Multi-class (9 types) | ~95%+ | — |

**Key Learnings:**
- IsolationForest detects **unknown** threats without labels — ideal for zero-day attacks
- Random Forest provides high-accuracy classification when labeled data is available
- Feature engineering (flag scores, ratios) dramatically improves detection accuracy
- Alert severity triage mirrors real SOC escalation workflows (CRITICAL → HIGH → MEDIUM)

**Industry Relevance:**  
This pipeline mirrors what commercial IDS/IPS tools like **Suricata**, **Zeek/Bro**, and **Darktrace** implement — a multi-layered detection stack combining rule-based, statistical, and ML approaches.